In [1]:
from neuron import h
import networkx as nx
import pygraphviz as gv

_ = h.load_file("RGCmodelGD.hoc")
RGC = h.DSGC(0, 0)
soma = RGC.soma
all_dends = RGC.dend

def dist_graph(rgc):
    """Node indices correspond to (dend index + 1) as the soma is the 0th node. e.g. RGC.dend[0] is node 1
    on the graph. Distances are between the centres (0.5) of each section corresponding to synapse locations."""
    g = nx.Graph()
    
    secs = [rgc.soma] + [d for d in rgc.dend]  # soma takes 0th position, offseting all dends by 1
    edges = []
    for i, parent in enumerate(secs):
        parent_ref = h.SectionRef(parent)
        for j in range(parent_ref.nchild()):
            child = parent_ref.child[j]
            # dend names are formated as "DSGC[1].dend[17]" and the id corresponds to the index
            # in the rgc.dend list of all dendrites
            cid = int(child.name().split(".")[1][5:-1]) + 1  # offset to accomodate 0th soma node
            # use end pos for soma rather than centre
            edges.append((i, cid, h.distance(parent(0.5 if i else 1.0), child(0.5))))
    g.add_weighted_edges_from(edges)
    return g
    
dg = dist_graph(RGC)

In [2]:
ego = nx.generators.ego_graph(dg, 0, radius=50, distance="weight")

In [3]:
adg = nx.nx_agraph.to_agraph(dg)

In [4]:
adg.draw("dist_graph_test.svg", prog="neato")

In [5]:
nx.nx_agraph.to_agraph(ego).draw("dist_graph_test_ego.svg", prog="neato")

In [6]:
# checking against dist matrix
# rr = np.random.default_rng()
# wrong = 0
# for _ in range(200):
#     a, b = rr.integers(2, 300, size=2)
#     gd = nx.shortest_path_length(dg, weight="weight", source=a, target=b)
#     md = rec_dists[(a - 1) * 2, (b - 1) * 2]
#     if gd != md:
#         diff = gd - md
#         # print("%.2f - %.2f = %.2f" % (gd, md, diff))
#         if abs(diff) > 20:
#             wrong += 1
# print("%i wrong" % wrong)